In [128]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt 
import plotly_express as px 
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

In [129]:
data = yf.download("AAPL", start='2020-01-01', end='2026-04-15')

[*********************100%***********************]  1 of 1 completed


In [130]:
data.head()

Price,Close,High,Low,Open,Volume
Ticker,AAPL,AAPL,AAPL,AAPL,AAPL
Date,,,,,
2020-01-02,72.400513,72.460776,71.156674,71.409778,135480400
2020-01-03,71.696640,72.455958,71.472462,71.629145,146322800
2020-01-06,72.267937,72.306506,70.568510,70.819208,118387200
2020-01-07,71.928055,72.533095,71.708695,72.277578,108872000
2020-01-08,73.085091,73.386408,71.631537,71.631537,132079200


In [131]:
data.columns = data.columns.droplevel(1)

In [132]:
data.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1578 entries, 2020-01-02 to 2026-04-14
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Close   1578 non-null   float64
 1   High    1578 non-null   float64
 2   Low     1578 non-null   float64
 3   Open    1578 non-null   float64
 4   Volume  1578 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 74.0 KB


In [133]:
# Calculating Price Change
data['Price_Change'] = data['Close'].diff()
data['pct_change'] = data['Close'].pct_change() * 100
data.fillna(0,inplace=True)
data.head(5)

Price,Close,High,Low,Open,Volume,Price_Change,pct_change
Date,,,,,,,
2020-01-02,72.400513,72.460776,71.156674,71.409778,135480400,0.000000,0.000000
2020-01-03,71.696640,72.455958,71.472462,71.629145,146322800,-0.703873,-0.972193
2020-01-06,72.267937,72.306506,70.568510,70.819208,118387200,0.571297,0.796825
2020-01-07,71.928055,72.533095,71.708695,72.277578,108872000,-0.339882,-0.470308
2020-01-08,73.085091,73.386408,71.631537,71.631537,132079200,1.157036,1.608602


In [134]:
data.columns

Index(['Close', 'High', 'Low', 'Open', 'Volume', 'Price_Change', 'pct_change'], dtype='object', name='Price')

In [135]:
# Seprating Gain and loss
data["Gain"] = data["Price_Change"].clip(lower=0)
data["Loss"] = -data["Price_Change"].clip(upper=0)

In [136]:
# Rolling Average
data['Avg_Gain'] = data['Gain'].rolling(5).mean()
data['Avg_Loss'] = data['Loss'].rolling(5).mean()

In [137]:
# Relative Strength
data['RS'] = data['Avg_Gain'] / data['Avg_Loss']

In [138]:
# RSI 
data['RSI'] = 100 - (100/(1 + data['RS']))

In [139]:
# Signal
data["Signal"] = "HOLD"
data.loc[(data["RSI"] < 30) & (data["RSI"].shift(1) >= 30), "Signal"] = "BUY"
data.loc[(data["RSI"] > 70) & (data["RSI"].shift(1) <= 70), "Signal"] = "SELL"

In [140]:
data.dropna(inplace=True)

In [141]:
data.head(21)

Price,Close,High,Low,Open,Volume,Price_Change,pct_change,Gain,Loss,Avg_Gain,Avg_Loss,RS,RSI,Signal
Date,,,,,,,,,,,,,,
2020-01-08,73.085091,73.386408,71.631537,71.631537,132079200,1.157036,1.608602,1.157036,-0.000000,0.345667,0.208751,1.655880,62.347699,HOLD
2020-01-09,74.637474,74.830314,73.810661,74.061352,170108400,1.552383,2.124077,1.552383,-0.000000,0.656143,0.208751,3.143187,75.863991,SELL
2020-01-10,74.806229,75.370301,74.304840,74.871318,140644800,0.168755,0.226099,0.168755,-0.000000,0.689894,0.067976,10.149027,91.030607,HOLD
2020-01-13,76.404404,76.430923,75.003882,75.122003,121532000,1.598175,2.136420,1.598175,-0.000000,0.895270,0.067976,13.170307,92.942990,HOLD
2020-01-14,75.372726,76.551483,75.249794,76.341768,161954400,-1.031677,-1.350285,0.000000,1.031677,0.895270,0.206335,4.338904,81.269565,HOLD
2020-01-15,75.049706,76.052490,74.618217,75.172645,121923600,-0.323021,-0.428565,0.000000,0.323021,0.663863,0.270940,2.450223,71.016369,HOLD
2020-01-16,75.989792,76.100682,75.230474,75.592055,108829200,0.940086,1.252618,0.940086,-0.000000,0.541403,0.270940,1.998243,66.647132,HOLD
2020-01-17,76.831100,76.833506,75.931967,76.238103,137816400,0.841309,1.107134,0.841309,-0.000000,0.675914,0.270940,2.494703,71.385267,SELL
2020-01-21,76.310402,76.900979,76.172999,76.459854,110843200,-0.520699,-0.677718,0.000000,0.520699,0.356279,0.375079,0.949876,48.714696,HOLD


In [142]:

data["Target"] = np.where(data["RSI"] < 30, "BUY",
                 np.where(data["RSI"] > 70, "SELL", "HOLD"))

In [143]:
features = ["RSI", "Avg_Gain", "Avg_Loss", "Price_Change", "pct_change"]

X = data[features]
Y = data["Target"]

In [144]:
# Train test Split 
split = int(len(data)*0.8)

X_train = X[:split]
X_test = X[split:]

Y_train = Y[:split]
Y_test = Y[split:]

In [145]:
# Scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [146]:
# LogisticRegression 
model_lr = LogisticRegression()
model_lr.fit(X_train,Y_train)

# Prediction
y_pred_lr = model_lr.predict(X_test)

# Accuracy Check
print('Accutacy :', accuracy_score(Y_test,y_pred_lr))

Accutacy : 0.9904761904761905


In [147]:
data.head()

Price,Close,High,Low,Open,Volume,Price_Change,pct_change,Gain,Loss,Avg_Gain,Avg_Loss,RS,RSI,Signal,Target
Date,,,,,,,,,,,,,,,
2020-01-08,73.085091,73.386408,71.631537,71.631537,132079200,1.157036,1.608602,1.157036,-0.000000,0.345667,0.208751,1.655880,62.347699,HOLD,HOLD
2020-01-09,74.637474,74.830314,73.810661,74.061352,170108400,1.552383,2.124077,1.552383,-0.000000,0.656143,0.208751,3.143187,75.863991,SELL,SELL
2020-01-10,74.806229,75.370301,74.304840,74.871318,140644800,0.168755,0.226099,0.168755,-0.000000,0.689894,0.067976,10.149027,91.030607,HOLD,SELL
2020-01-13,76.404404,76.430923,75.003882,75.122003,121532000,1.598175,2.136420,1.598175,-0.000000,0.895270,0.067976,13.170307,92.942990,HOLD,SELL
2020-01-14,75.372726,76.551483,75.249794,76.341768,161954400,-1.031677,-1.350285,0.000000,1.031677,0.895270,0.206335,4.338904,81.269565,HOLD,SELL


In [148]:
result = pd.DataFrame({
    'Actual': Y_test,
    'Prediction': y_pred_lr
})
result.head()

,Actual,Prediction
Date,,
2025-01-10,BUY,BUY
2025-01-13,BUY,BUY
2025-01-14,BUY,BUY
2025-01-15,HOLD,HOLD
2025-01-16,BUY,BUY


In [149]:
X_test = pd.DataFrame(X_test, index=Y_test.index)

In [150]:
data.loc[X_test.index, 'ML_Signal'] = y_pred_lr

In [151]:
data['ML_Signal'] = data['ML_Signal'].map({1: 'BUY', 0: 'SELL'})

In [152]:
# Final Signal
def final_signal(row):
    if row['Signal'] == 'BUY' and row['ML_Signal'] == 'BUY':
        return 'STRONG BUY'
    elif row['Signal'] == 'SELL' and row['ML_Signal'] == 'SELL':
        return 'STRONG SELL'
    else:
        return 'HOLD'
    
data['Final_Signal'] = data.apply(final_signal, axis = 1)